# **Designing Data-Intensive Applications - notes**

## Chapter 1 Highlights

#### **Twitter's querying methods, for efficiency**

pg.11-13

Twitter’s scaling problem has to do with fan-out (term borrowed from EE, where it describes the number of logic gate inputs that are attached to another gate’s output. The output needs to supply enough current to drive all the attached inputs. In transaction processing systems, we use it to describe the number of requests to other services that we need to make in order to serve one incoming request.) (new query every time you load up home page, used for large celebrities)

Twitter caches the tweets of each user and automatically sends it to the home timeline table of all of their followers. This is a lot of computations per tweet, especially for a frequent tweeter with a lot of followers. So they’ve combined their old school method, a SQL command that renders the tweets of the people the current_user follows when the current_user loads up their home page, with the newer method involving caching for larger pages/accounts, such as celebrities and/or politicians. (caches homepage of every user... “when a user posts a tweet, look up all of their followers and put the tweet in their home pages”) (how most tweets are handled).

If you applied that first method of writing to all people following you, writes for celebrities/politicians/etc. get expensive. So, the first method (fan-out on write) is for normal/smaller accounts, while larger accounts only need to write once and then merge into the fan-out on read (timeline).

Twitter currently uses a combo of both methods.


#### **"Averages lie"**

Outliers from your users can skew the data so much that the averages are no longer good representation of efficiency/run time/etc. **So, instead we use percentiles** (p50 = median (50% of users), p99 (1 in 100), p99.9 (1 in 1000 and takes the longest)). 





## ***Chapter 1: other notes for myself***

**New terms to me, from this chapter:** Sharding, eventual consistency, ACID (forgot), CAP theorem, MapReduce (forgot), MTTF (Mean Time to Failure), RAID (Redundant Array of Independent Disks, forgot), 


**Intereting topics from this chapter:**
- Free and open source software is now preferred 
- “CPU clock speeds are barely increasing, but multi-core processors are standard and networks are getting faster. This means parallelism is only going to increase.”
- AWS as IaaS was a game changer
- Data-intensive vs computer-intensive


1. **Reliability:** tolerating hardware and software faults, human error
2. **Scalability:** measuring load and performance, latency percentiles, throughput, as the system grows there should be ways of dealing with that growth without falling a part
3. **Maintainability:** operability, simplicity, and evolvability, easy for developers to modify, operate and understand


**Examples of some of the needs of data-intensive applications:**
- Store data so that they or another application can find it again later (databases)
- Remember the result of an expensive operation, to speed up reads (cycles)
- Allow users to search data by keyword or filter it in various ways (search indexes)
- Send a message to another process, to be handled asynchronously (stream processing)
- Periodically crunch a large amount of accumulated data (batch processing)

**Redis** - datastore that can be used as message queues
**Apache Kafka** - message queue that has some database-like durability guarantees
**Memcached** - application-managed caching layer
**Elasticsearch, Solr** - full-text search servers


**Chaos Monkey** is a resiliency tool developed by Netflix that randomly terminates virtual machine instances and containers in its production environment to test the system's ability to withstand failures, a core practice of chaos engineering. **It's part of a larger suite called the **Simian Army****, designed to intentionally introduce faults to find weaknesses and build more resilient, fault-tolerant systems, ensuring Netflix services remain stable even when components fail.


https://en.wikipedia.org/wiki/Leap_second

https://www.wired.com/2012/07/leap-second-glitch-explained/


Describing performance:

When load increases:
- How is performance affected?
- How much do you need to increase resources if you want to keep performance the same?

Batch processing systems like Hadoop (Apache Hadoop, open source framework used to store/process massive datasets across groups of ordinary computers… relies on HDFS, YARN, and MapReduce), we usually care about throughput (number of records processed per second).

Latency: time delay between a user's request and a system's response
_______________________
P. 14

**Response time:** what the client sees, service time+network delays+queueing delays

**Tail latencies:** high percentiles of response times

**Service level agreement (SLA) and service level objectives (SLO):** contracts that define the expected performance and availability of a service

Queueing delays often account for a large part of the response time at high percentiles. As a server can only process a small number of things in parallel (limited, for example, by its number of CPU cores), it only takes a small number of slow requests to hold up the processing of subsequent requests… this is called head-of-line blocking

**Tail latency amplification:** the chances of getting a slow call increases if an end-user request requires multiple back end calls, and so a higher proportion of end-user requests end up being slow


**Monitoring response times:** naive implementation = keeping a list of response times for all requests within a time window and sort the list every minute; algorithms to use instead of naive: forward decay, t-digest, HdrHistogram

(you cannot average percentiles (check the Prometheus docs): https://www.youtube.com/shorts/O_txxXkxsfQ) 

_______________________
**Scaling up:** vertical scaling, moving to a more powerful machine
**Scaling out:** horizontal scaling, distributing the load across multiple smaller machines

**Shared-nothing Architecture:** distributing load across multiple machines

**Elastic systems/architecture:** systems that automatically add computing resources when they detect a load increase, whereas other systems are scaled manually (a human analyzes the capacity and decides to add more machines to the system). These systems can be useful if load is highly unpredictable, but manually scaled systems are simpler and may have fewer operational surprises. (pg. 17)

_______________________

“While distributing stateless services (programs that process on its own without saving any information from past interactions) across multiple machines is fairly straightforward, taking stateful data systems from a single node to distribute setup can introduce a lot of additional complexity.” p. 18… common wisdom until recently was to keep database on a single node (scale up) until scaling costs/requirements forced you to make it distributed


The three design principles for software systems:
- **Operability:** make it easy for operations teams to keep the systems running smoothly
- **Simplicity**: make it easy for new engineers to understand the system, by removing as much complexity as possible from the system
- **Evolvability**: make it easy for engineers to make changes to the system in the future, adapting it for unanticipated use cases as requirements change (aka, extensibility, modifiability, or plasticity)

